In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file into environment variables

API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [2]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'setup working' and nothing else."}]
)
print(resp.choices[0].message.content)

setup working


In [3]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content
answer = ask_llm("What is the capital of Ghana?")
print(answer)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Ghana?"},
    ],
)
print(response.usage)

The capital of Ghana is Accra.
CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041190898, prompt_time=0.001443055, completion_time=0.012300796, total_time=0.013743851)


1. System is the set of instructions that define how a model should behave before the conversation begins while user is the actual task you ask a model in the moment.
Examples:
System: You are a loan assistant, hence use only facts from data supplied to you.
User: Summarize the document pasted below, include only important and urgent information in the summary.
2. Token is basically a small piece of text from a corpus that a model processes.
3. API provider charge per token because request can be widely different in size,computational and analytical cost. Charging per token requires a payment on how much text you actually used, thereby creating fairness.


In [7]:
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature 0.0")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}")

print("\n=== Temperature 1.2 ===")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}")

Temperature 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **TradeUp Savings**: This name suggests that the savings product will help traders "trade up" and improve their financial situation.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could convey the idea of gathering savings.
4. **MarketMoen**: This name combines "market" and "moen" (a play on the word "money"), which could appeal to market traders.
5. **Adanfo Savings**: "Adanfo" means "helpers" or "supporters" in the Akan language, which could suggest that the savings product is a helpful tool for market traders.
6. **Kae Savings**: "Kae" means "grow" or "increase" in the Ga language, which is spoken in the Accra region. This name could convey the idea of growing one's savings.

Th

1. At temperature 0.0, there were some repetition(Makola, Accra Market, Savings,etc) in the answers across all 5 runs wjile at Temperature 1.2, the answers were quite varied, new words and ideas were introduced at each run. Therefore, for the loan decision support system, low temperature would be more suitable because the system needs to produce consistent and factual summaries that are very close to the application givint to the model. However, temperature alone doesnt guarantee factual accuracy nor prevent hallucination, hence the need for human oversight.

In [9]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.
